In [17]:
from prebuilt_tools import get_tool_by_id

def _prebuilt_placeholder(tool_data):
    return get_tool_by_id(tool_data.get("id"))

def _custom_function_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {"message": "Custom function placeholder", "tool": tool_data.get("name")}
    return _run

In [18]:
from langchain_core.tools import StructuredTool
from pydantic import create_model
import requests
from typing import Dict, Any, List

def _custom_api_placeholder(tool_def: Dict[str, Any]) -> List:
    """
    Convert custom_api tool definition to LangChain tool
    
    Returns:  List containing one LangChain StructuredTool
    """
    name = tool_def["name"]
    description = tool_def["description"]
    api_url = tool_def["api_url"]
    api_request_type = tool_def["api_request_type"]
    custom_message = tool_def.get("custom_message", "")
    input_schema = tool_def["input_schema"]
    
    # Build Pydantic model from input_schema
    fields = {}
    properties = input_schema.get("properties", {})
    
    for field_name, field_spec in properties.items():
        field_type_map = {
            "string": str,
            "number": float,
            "integer":  int,
            "boolean": bool
        }
        field_type = field_type_map.get(field_spec.get("type"), str)
        fields[field_name] = (field_type, None)
    
    # Fallback if no properties
    if not fields: 
        fields = {"_placeholder": (str, None)}
    
    InputModel = create_model(f"{name}_Input", **fields)
    
    # API call function
    def execute_api_call(**kwargs) -> Dict[str, Any]:
        try:
            # Remove placeholder if exists
            kwargs.pop("_placeholder", None)
            
            if api_request_type.upper() == "GET":
                resp = requests.get(api_url, params=kwargs, timeout=10)
            else:  # POST
                resp = requests.post(api_url, json=kwargs, timeout=10)
            
            return {
                "status_code": resp.status_code,
                "data": resp.json() if resp.content else {},
                "custom_message": custom_message
            }
        except Exception as e:
            return {
                "status_code": 500,
                "data": {"error":  str(e)},
                "custom_message": custom_message
            }
    
    # Create LangChain tool
    tool = StructuredTool.from_function(
        func=execute_api_call,
        name=name,
        description=description,
        args_schema=InputModel
    )
    
    return [tool]

In [19]:
from typing import List
from langchain_core.tools import StructuredTool
from manager import ToolRegistryManager

def build_langchain_tools(tool_ids: List[str]) -> List[StructuredTool]:
    """
    Given a list of tool IDs, return LangChain-compatible Tool objects.
    """
    manager = ToolRegistryManager()
    langchain_tools: List[StructuredTool] = []

    for tool_id in tool_ids:
        tool_data = manager.get_tool(tool_id)
        if not tool_data:
            print("tool is missing")
            continue

        tool_type = tool_data.get("type")
        name = tool_data.get("name")
        description = tool_data.get("description")

        # --- Create tools based on type ---
        if tool_type == "prebuilt":
            tool = _prebuilt_placeholder(tool_data)
            langchain_tools.append(tool)

        elif tool_type == "custom_function":
            func = _custom_function_placeholder(tool_data)
            tool = StructuredTool.from_function(
                func=func,
                name=name,
                description=description
            )
            langchain_tools.append(tool)

        elif tool_type == "custom_api":
            # _custom_api_placeholder returns a list of tools
            tools = _custom_api_placeholder(tool_data)
            langchain_tools.extend(tools)

    return langchain_tools

In [20]:
tool_ids = [
    "tool_517087cd-4f45-4dfb-835d-ec908086baa4",
    "tool_9b2d94e3-1c6e-4f59-91a1-61e1cc0a6db1",
    "tool_c78a0e62-b6c1-49cf-9e5b-33f2cde54a77",
    "tool_2aee9e7a-7d67-4f13-9f91-bd7bb91e84fd",
    "tool_fe925649-e6ec-4002-8d7f-7374d07ffa7d",
    "tool_54da582b-e8e6-4f98-ab19-bfaf0a5965b6"
]

tools = build_langchain_tools(tool_ids)

In [23]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()


llm = ChatOpenAI(temperature=0)

agent = create_agent(model=llm, tools=tools)

In [26]:
response1 = agent.invoke({"messages": [("user", "Is john.doe@example.com a valid email?")]})
response2 = agent.invoke({"messages": [("user", "Validate this phone number: 9876543210")]})
response3 = agent.invoke({"messages": [("user", "Check the strength of password: MyP@ssw0rd123")]})
response4 = agent.invoke({"messages": [("user", "Is this URL valid: https://www.example.com/page")]})
response5 = agent.invoke({"messages": [("user", "Get details for order ID: ORD12345")]})
response6 = agent.invoke({"messages": [("user", "Create a new order with ID NEW001 for item: laptop")]})
response7 = agent.invoke({"messages": [("user", "Validate email test@domain.com and phone 9123456789")]})
response8 = agent.invoke({"messages": [("user", "Check if invalid@email is valid")]})

In [29]:
# Extract tool names from responses
def extract_tools_invoked(response):
    """Extract tool names invoked in a response"""
    tools = []
    messages = response.get('messages', [])
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tool_call in msg.tool_calls:
                tools.append(tool_call['name'])
    return tools

# Store all responses
responses = [response1, response2, response3, response4, response5, response6, response7, response8]

# Create a summary of all responses
responses_summary = {}
for i, resp in enumerate(responses, 1):
    tools_invoked = extract_tools_invoked(resp)
    responses_summary[f"response_{i}"] = tools_invoked

# Define required tools for each test
required_tools = {
    "response_1": ["email_validator"],
    "response_2": ["phone_validator"],
    "response_3": ["password_strength_checker"],
    "response_4": ["url_validator"],
    "response_5": ["order_info"],
    "response_6": ["create_order"],
    "response_7": ["email_validator", "phone_validator"],
    "response_8": ["email_validator"]
}

# Check if required tools were invoked
print("TOOL INVOCATION CHECK")
print("=" * 50)

for response_id, required in required_tools.items():
    invoked = responses_summary[response_id]
    is_complete = set(required) == set(invoked)
    status = "✓ PASS" if is_complete else "✗ FAIL"
    print(f"{response_id}: {status}")
    if not is_complete:
        print(f"  Expected: {required}")
        print(f"  Got: {invoked}")

TOOL INVOCATION CHECK
response_1: ✓ PASS
response_2: ✓ PASS
response_3: ✓ PASS
response_4: ✓ PASS
response_5: ✓ PASS
response_6: ✓ PASS
response_7: ✓ PASS
response_8: ✓ PASS


In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from typing import Dict

class FieldPromptInput(BaseModel):
    field: str = Field(..., description="The field value to validate")
    prompt: str = Field(..., description="Description of the expected data type/format (e.g., 'email address', 'phone number', 'date in YYYY-MM-DD format')")

class ValidationResult(BaseModel):
    valid: bool = Field(..., description="Whether the field matches the expected format")
    score: int = Field(..., description="A validation score from 0 to 100")

def field_prompt_validator_func(field: str, prompt: str) -> Dict:
    """
    Validates whether a field value matches the data type/format specified in the prompt.
    
    Args:
        field: The actual value to validate
        prompt: Description of expected data type/format
        
    Returns:
        Dictionary with validation results including score, validity, and reason
    """
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    validation_prompt = f"""You are a data validation expert. Analyze whether the given field value matches the expected data type or format.

    Field Value: "{field}"
    Expected Format/Type: {prompt}

    Evaluate:
    1. Does the field value match the expected format/type?
    2. Provide a validation score from 0 to 100 (0 = completely invalid, 100 = perfectly valid)
    3. Explain your reasoning

    Respond in JSON format with:
    - "valid": true/false
    - "score": 0-100
    - "reason": brief explanation
    """
    
    try:
        response = llm.invoke(validation_prompt)
        
        # Parse the response content
        import json
        result = json.loads(response.content)
        
        return {
            "valid": result.get("valid", False),
            "score": result.get("score", 0),
        }
    except Exception as e:
        return {
            "valid": False,
            "score": 0,
        }

field_prompt_validator = StructuredTool.from_function(
    name="field_prompt_validator",
    description="Validates whether a field value matches the data type or format specified in the prompt. Takes a field value and a description of the expected format, returns validation score and reasoning.",
    func=field_prompt_validator_func,
    args_schema=FieldPromptInput,
)

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from typing import Dict
import json
import re

class FieldPromptInput(BaseModel):
    field: str = Field(..., description="The field value to validate")
    prompt: str = Field(..., description="Description of the expected data type/format (e.g., 'email address', 'phone number', 'date in YYYY-MM-DD format')")

class ValidationResult(BaseModel):
    valid: bool = Field(..., description="Whether the field matches the expected format")
    score: int = Field(..., description="A validation score from 0 to 100")

def field_prompt_validator_func(field: str, prompt: str) -> Dict:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    validation_prompt = f"""You are a data validation expert. Analyze whether the given field value matches the expected data type or format.

    Field Value: "{field}"
    Expected Format/Type: {prompt}

    Evaluate:
    1. Does the field value match the expected format/type?
    2. Provide a validation score from 0 to 100 (0 = completely invalid, 100 = perfectly valid)
    3. Explain your reasoning

    Respond ONLY with valid JSON in this exact format (no markdown, no code blocks):
    {{"valid": true, "score": 100, "reason": "explanation here"}}"""

    try:
        response = llm.invoke(validation_prompt)
        content = response.content.strip()
        
        # Remove markdown code blocks if present
        if content.startswith("```"):
            content = re.sub(r'```(?:json)?\n?', '', content)
            content = content.strip()
        
        # Parse JSON
        result = json.loads(content)
        
        return {
            "valid": result.get("valid", False),
            "score": result.get("score", 0),
        }
    except json.JSONDecodeError as e:
        print(f"JSON Parse Error: {e}")
        print(f"Response content: {response.content}")
        return {
            "valid": False,
            "score": 0,
        }
    except Exception as e:
        print(f"Error: {e}")
        return {
            "valid": False,
            "score": 0,
        }

field_prompt_validator = StructuredTool.from_function(
    name="field_prompt_validator",
    description="Validates whether a field value matches the data type or format specified in the prompt. Takes a field value and a description of the expected format, returns validation score and reasoning.",
    func=field_prompt_validator_func,
    args_schema=FieldPromptInput,
)

# ========================================
# TEST CASES FOR EMAIL FORMAT
# ========================================

# Test 1: Valid email
result1 = field_prompt_validator.invoke({
    "field": "john.doe@example.com",
    "prompt": "email address"
})
print("Test 1 - Valid Email:")
print(f"  Field: john.doe@example.com")
print(f"  Valid: {result1['valid']}")
print(f"  Score: {result1['score']}")
print()

# Test 2: Invalid email (missing @)
result2 = field_prompt_validator.invoke({
    "field": "invalidemail.com",
    "prompt": "email address"
})
print("Test 2 - Invalid Email (missing @):")
print(f"  Field: invalidemail.com")
print(f"  Valid: {result2['valid']}")
print(f"  Score: {result2['score']}")
print()

# Test 3: Invalid email (missing domain)
result3 = field_prompt_validator.invoke({
    "field": "user@",
    "prompt": "email address"
})
print("Test 3 - Invalid Email (missing domain):")
print(f"  Field: user@")
print(f"  Valid: {result3['valid']}")
print(f"  Score: {result3['score']}")
print()

# Test 4: Valid email with subdomain
result4 = field_prompt_validator.invoke({
    "field": "admin@mail.company.com",
    "prompt": "email address"
})
print("Test 4 - Valid Email (with subdomain):")
print(f"  Field: admin@mail.company.com")
print(f"  Valid: {result4['valid']}")
print(f"  Score: {result4['score']}")
print()

# Test 5: Invalid email (spaces)
result5 = field_prompt_validator.invoke({
    "field": "user name@example.com",
    "prompt": "email address"
})
print("Test 5 - Invalid Email (contains spaces):")
print(f"  Field: user name@example.com")
print(f"  Valid: {result5['valid']}")
print(f"  Score: {result5['score']}")

Test 1 - Valid Email:
  Field: john.doe@example.com
  Valid: True
  Score: 100

Test 2 - Invalid Email (missing @):
  Field: invalidemail.com
  Valid: False
  Score: 0

Test 3 - Invalid Email (missing domain):
  Field: user@
  Valid: False
  Score: 10

Test 4 - Valid Email (with subdomain):
  Field: admin@mail.company.com
  Valid: True
  Score: 100

Test 5 - Invalid Email (contains spaces):
  Field: user name@example.com
  Valid: False
  Score: 50


In [6]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Dict
import base64

client = OpenAI()

class FilePromptInput(BaseModel):
    pdf_path: str = Field(..., description="Path to the PDF document")
    prompt: str = Field(..., description="Prompt to analyze the PDF")

class ValidationResult(BaseModel):
    score: int = Field(..., description="A validation score from 0 to 100")

def file_prompt_validator_func(pdf_path: str, prompt: str) -> ValidationResult:
    print("the file validator tool is invoked")
    
    with open(pdf_path, "rb") as f:
        data = f.read()
    
    base64_string = base64.b64encode(data).decode("utf-8")

    response = client.responses.create(
            model="gpt-5",
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_file",
                            "filename": "document.pdf",
                            "file_data": f"data:application/pdf;base64,{base64_string}",
                        },
                        {
                            "type": "input_text",
                            "text": f"you are validation agent. validate the file provided based on this {prompt} and provide the score.",
                        },
                    ],
                },
            ]
        )

    print(f" the fiel_prompt_validator tool response: {response}")
    return response

file_prompt_validator = StructuredTool.from_function(
    name="file_prompt_validator",
    description="Analyzes a PDF docuement based on a given prompt and returns a validation score.",
    func=file_prompt_validator_func,
    args_schema= FilePromptInput,
)

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Dict

client = OpenAI()

class FilePromptInput(BaseModel):
    pdf_path: str = Field(..., description="Path to the PDF document")
    prompt: str = Field(..., description="Prompt to analyze the PDF")

class ValidationResult(BaseModel):
    valid: bool = Field(..., description="Whether the validation passed")
    score: int = Field(..., description="A validation score from 0 to 100")
    
def file_prompt_validator_func(pdf_path: str, prompt: str) -> Dict:
    print("the file validator tool is invoked")
    
    import base64
    with open(pdf_path, "rb") as f:
        data = f.read()
    
    base64_string = base64.b64encode(data).decode("utf-8")

    response = client.responses.create(
        model="gpt-5",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{base64_string}",
                    },
                    {
                        "type": "input_text",
                        "text": f"You are a validation agent. Validate the file provided based on this: {prompt}. Provide a score from 0-100.",
                    },
                ],
            },
        ]
    )
    
    # Extract the text from the response
    output_message = None
    for item in response.output:
        if item.type == 'message':
            output_message = item
            break
    
    if output_message and output_message.content:
        # Get the text content
        text_content = ""
        for content_item in output_message.content:
            if content_item.type == 'output_text':
                text_content = content_item.text
                break
        
        print(f"Validation response: {text_content}")
        
        import re
        score_match = re.search(r'(\d+(?:\.\d+)?)\s*/\s*10', text_content)
        score = 0
        if score_match:
            score = int(float(score_match.group(1)) * 10)
        
        return {
            "valid": score >= 70,  # Consider valid if score >= 70
            "score": score,
            "reason": text_content[:500]  # First 500 chars as reason
        }
    
    return {
        "valid": False,
        "score": 0,
        "reason": "Failed to extract response"
    }

file_prompt_validator = StructuredTool.from_function(
    name="file_prompt_validator",
    description="Analyzes a PDF document based on a given prompt and returns a validation score.",
    func=file_prompt_validator_func,
    args_schema=FilePromptInput,
)

In [34]:
response = file_prompt_validator.invoke({
    "pdf_path": r"C:\github_work\cenomi-platform-agents\test_doc.pdf",
    "prompt": "check whether the document contains the data pipelines"
})

the file validator tool is invoked
Validation response: 95


In [ ]:
response.output[-1].content[0].text

In [2]:
import requests
from typing import Optional

def upload_document(
    file_path: str,
    uri: str = "http://20.224.157.137:8000/v1/documents",
    file_extension: str = "pdf",
    request_id: str = "test-request-123",
    pms_id: str = "96225",
    pms_tenant_id: str = "t0108240",
    pms_customer_id: str = "customer-code",
    document_type_id: str = "FIT_ARCH_DRW",
    process_type_id: str = "2000",
    source: str = "tenant_central",
    revised_version: str = "no",
    cenomi_contact_name: str = "Test Contact",
    cenomi_contact_role: str = "Test Role"
) -> dict:
    """
    Upload a document to the specified API endpoint with multipart/form-data.
    
    Args:
        file_path: Path to the PDF file to upload
        uri: API endpoint URL
        file_extension: File extension (default: "pdf")
        request_id: Request identifier
        pms_id: PMS ID
        pms_tenant_id: PMS Tenant ID
        pms_customer_id: PMS Customer ID
        document_type_id: Document type identifier
        process_type_id: Process type identifier
        source: Source system
        revised_version: Whether this is a revised version ("yes" or "no")
        cenomi_contact_name: Contact person name
        cenomi_contact_role: Contact person role
    
    Returns:
        dict: Response from the API
    """
    # Prepare the files for upload
    with open(file_path, 'rb') as f:
        files = {
            'document': (file_path.split('\\')[-1], f, 'application/pdf')
        }
        
        # Prepare the form data
        data = {
            'file_extension': file_extension,
            'request_id': request_id,
            'pms_id': pms_id,
            'pms_tenant_id': pms_tenant_id,
            'pms_customer_id': pms_customer_id,
            'document_type_id': document_type_id,
            'process_type_id': process_type_id,
            'source': source,
            'revised_version': revised_version,
            'cenomi_contact_name': cenomi_contact_name,
            'cenomi_contact_role': cenomi_contact_role
        }
        
        # Make the POST request
        try:
            response = requests.post(uri, files=files, data=data)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            return {
                "error": str(e),
                "status_code": response.status_code if hasattr(response, 'status_code') else None
            }

In [4]:
result = upload_document(
    file_path=r"C:\github_work\cenomi-platform-agents\test_doc.pdf"
)
print(result)

{'success': True, 'data': {'document_id': '2924a1e9-e770-4110-8811-bb4f093fe6a9', 'file_name': 'test_doc.pdf', 'message': 'Document uploaded successfully'}}


In [ ]:
import requests
from typing import Optional

def upload_document(
    file_path: str,
    file_extension: str,
    request_id: str,
    pms_id: str,
    pms_tenant_id: str,
    pms_customer_id: str,
    document_type_id: str,
    process_type_id: str,
    source: str,
    revised_version: str,
    cenomi_contact_name: str,
    cenomi_contact_role: str,
    uri: str = "http://20.224.157.137:8000/v1/documents",
) -> dict:
    # Prepare the files for upload
    with open(file_path, 'rb') as f:
        files = {
            'document': (file_path.split('\\')[-1], f, 'application/pdf')
        }
        
        # Prepare the form data
        data = {
            'file_extension': file_extension,
            'request_id': request_id,
            'pms_id': pms_id,
            'pms_tenant_id': pms_tenant_id,
            'pms_customer_id': pms_customer_id,
            'document_type_id': document_type_id,
            'process_type_id': process_type_id,
            'source': source,
            'revised_version': revised_version,
            'cenomi_contact_name': cenomi_contact_name,
            'cenomi_contact_role': cenomi_contact_role
        }
        
        # Make the POST request
        try:
            response = requests.post(uri, files=files, data=data)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            return {
                "error": str(e),
                "status_code": getattr(e.response, 'status_code', None) if hasattr(e, 'response') else None
            }

{'success': True, 'data': {'document_id': 'e5c85e00-9c4e-4e2c-a833-1d4a0fcaca2d', 'file_name': 'test_doc.pdf', 'message': 'Document uploaded successfully'}}


In [11]:
result = upload_document(
    file_path=r"C:\github_work\cenomi-platform-agents\test_doc.pdf",
    file_extension="pdf",
    request_id="test-request-123",
    pms_id="96225",
    pms_tenant_id="t0108240",
    pms_customer_id="customer-code",
    document_type_id="FIT_ARCH_DRW",
    process_type_id="2000",
    source="tenant_central",
    revised_version="no",
    cenomi_contact_name="Test Contact",
    cenomi_contact_role="Test Role"
)

print(result)

{'success': True, 'data': {'document_id': 'eb4a3700-7eaf-4ebf-b5fd-117d1f24a10d', 'file_name': 'test_doc.pdf', 'message': 'Document uploaded successfully'}}
